In [9]:
!pip install simpy

In [14]:
import simpy
import random

# temel ayarlam
RANDOM_SEED = 42
SIM_SURESI = 60          # İlk gösterim için 60 dakika (1 saat)
MUSTERI_GELIS_ARALIGI = 3 # Ortalama 3 dakikada bir yeni müşteri
random.seed(RANDOM_SEED)

In [15]:
class Market(object):
    def __init__(self, env, n_normal, n_hizli, n_dijital):
        self.env = env
        self.normal_kasa = simpy.Resource(env, n_normal)
        self.hizli_kasa = simpy.Resource(env, n_hizli)
        self.dijital_kasa = simpy.Resource(env, n_dijital)

    def odeme_yap(self, urun_sayisi):
        # Ürün başına 0.4 dakika işlem süresi
        yield self.env.timeout(urun_sayisi * 0.4)

def musteri(env, isim, market, bekleme_listesi, sessiz=False):
    gelis_zamani = env.now
    urun_sayisi = random.randint(1, 25)

    if not sessiz:
        print(f" {isim} | {env.now:.2f}. dk | Markete girdi ({urun_sayisi} ürün)")

    # 10 ürün ve altı Hızlı Kasa'ya, diğerleri %30 ihtimalle Dijital'e, kalanı Normal'e
    if urun_sayisi <= 10:
        secilen_kasa = market.hizli_kasa
        tip = "Hızlı Kasa"
    elif random.random() < 0.3:
        secilen_kasa = market.dijital_kasa
        tip = "Dijital Kasa"
    else:
        secilen_kasa = market.normal_kasa
        tip = "Normal Kasa"

    with secilen_kasa.request() as istek:
        yield istek
        bekleme = env.now - gelis_zamani
        bekleme_listesi.append(bekleme)

        if not sessiz:
            print(f"   --> {isim} | {tip} sırasına girdi. (Bekleme: {bekleme:.2f} dk)")

        yield env.process(market.odeme_yap(urun_sayisi))

        if not sessiz:
            print(f"      OK {isim} | Ödemeyi tamamladı: {env.now:.2f}. dk")

def sistem_akisi(env, n_normal, n_hizli, n_dijital, bekleme_listesi, sessiz=False):
    market = Market(env, n_normal, n_hizli, n_dijital)
    i = 0
    while True:
        yield env.timeout(random.expovariate(1.0 / MUSTERI_GELIS_ARALIGI))
        i += 1
        env.process(musteri(env, f"Müşteri {i}", market, bekleme_listesi, sessiz))

In [17]:
# (2 Normal Kasa Senaryosu)
print("="*60)
print("BÖLÜM 1: MARKET HAREKET GÜNLÜĞÜ")
print("="*60)
detayli_bekleme = []
env_detay = simpy.Environment()
env_detay.process(sistem_akisi(env_detay, 2, 1, 1, detayli_bekleme, sessiz=False))
env_detay.run(until=SIM_SURESI)

# OPTİMİZASYON TABLOSU
print("\n" + "="*60)
print("BÖLÜM 2: KASİYER SAYISI PERFORMANS ANALİZİ")
print("="*60)
print(f"{'Normal Kasiyer':<15} | {'Ortalama Bekleme Süresi'}")
print("-" * 45)

for k in range(1, 6):
    analiz_listesi = []
    random.seed(RANDOM_SEED) # Her seferinde aynı müşteri akışını test etmek için
    env_analiz = simpy.Environment()
    env_analiz.process(sistem_akisi(env_analiz, k, 1, 1, analiz_listesi, sessiz=True))
    env_analiz.run(until=120) # Daha sağlıklı analiz için 120 dakika test ediyoruz

    ortalama = sum(analiz_listesi) / len(analiz_listesi) if analiz_listesi else 0
    print(f"{k} Kasiyer{' '*6} | {ortalama:.2f} Dakika")

print("="*60)

BÖLÜM 1: MARKET HAREKET GÜNLÜĞÜ
 Müşteri 1 | 0.92. dk | Markete girdi (6 ürün)
   --> Müşteri 1 | Hızlı Kasa sırasına girdi. (Bekleme: 0.00 dk)
 Müşteri 2 | 3.00. dk | Markete girdi (4 ürün)
      OK Müşteri 1 | Ödemeyi tamamladı: 3.32. dk
   --> Müşteri 2 | Hızlı Kasa sırasına girdi. (Bekleme: 0.32 dk)
      OK Müşteri 2 | Ödemeyi tamamladı: 4.92. dk
 Müşteri 3 | 5.13. dk | Markete girdi (10 ürün)
   --> Müşteri 3 | Hızlı Kasa sırasına girdi. (Bekleme: 0.00 dk)
      OK Müşteri 3 | Ödemeyi tamamladı: 9.13. dk
 Müşteri 4 | 11.26. dk | Markete girdi (17 ürün)
   --> Müşteri 4 | Normal Kasa sırasına girdi. (Bekleme: 0.00 dk)
 Müşteri 5 | 16.79. dk | Markete girdi (25 ürün)
   --> Müşteri 5 | Dijital Kasa sırasına girdi. (Bekleme: 0.00 dk)
 Müşteri 6 | 17.29. dk | Markete girdi (17 ürün)
   --> Müşteri 6 | Normal Kasa sırasına girdi. (Bekleme: 0.00 dk)
      OK Müşteri 4 | Ödemeyi tamamladı: 18.06. dk
      OK Müşteri 6 | Ödemeyi tamamladı: 24.09. dk
 Müşteri 7 | 26.49. dk | Markete girdi